In [1]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd
import os
import matplotlib.pyplot as plt
import argparse
import numpy as np
from scipy import signal
from scipy.interpolate import interp1d


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\buck\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\buck\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\buck\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\buck\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\buck\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\buck\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



In [3]:

for dirname, _, filenames in os.walk('data/raw_gen'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

data/raw_gen\hr60.csv
data/raw_gen\hr87.csv
data/raw_gen\hr90.csv


In [4]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [11]:
# ==============================
# 1. Folder path
# ==============================

folder_path = "data\\raw_gen"

results = []

# ==============================
# 2. Clean RR
# ==============================

def clean_rr(rr):
    rr_clean = []
    for i in range(1, len(rr)):
        if 300 < rr[i] < 2000 and abs(rr[i] - rr[i-1]) < 0.2 * rr[i-1]:
            rr_clean.append(rr[i])
    return np.array(rr_clean)

# ==============================
# 3. Time-domain
# ==============================

def time_domain_features(nn):

    mean_rr = np.mean(nn)
    hr = 60000 / mean_rr
    sdnn = np.std(nn, ddof=1)

    diff_nn = np.diff(nn)
    rmssd = np.sqrt(np.mean(diff_nn**2))

    nn50 = np.sum(np.abs(diff_nn) > 50)
    pnn50 = 100 * nn50 / len(diff_nn) if len(diff_nn) > 0 else 0

    return mean_rr, hr, sdnn, rmssd, pnn50

# ==============================
# 4. Frequency-domain
# ==============================

def frequency_domain_features(nn):

    if len(nn) < 10:
        return 0, 0, 0

    t = np.cumsum(nn) / 1000.0
    t = t - t[0]

    fs = 4  # chuẩn HRV guideline
    f_interp = interp1d(t, nn, kind='cubic', fill_value="extrapolate")
    t_new = np.arange(0, t[-1], 1/fs)
    nn_interp = f_interp(t_new)

    nn_interp = nn_interp - np.mean(nn_interp)

    f, psd = signal.welch(nn_interp, fs=fs, nperseg=min(256, len(nn_interp)))

    lf_mask = (f >= 0.04) & (f < 0.15)
    hf_mask = (f >= 0.15) & (f < 0.4)

    lf = np.trapezoid(psd[lf_mask], f[lf_mask])
    hf = np.trapezoid(psd[hf_mask], f[hf_mask])

    lf_hf = lf / hf if hf > 0 else 0

    return lf, hf, lf_hf

# ==============================
# 5. Loop toàn bộ file
# ==============================

for filename in os.listdir(folder_path):

    if filename.endswith(".csv"):

        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path)

        # Lấy R-peaks (Peak = 3)
        r_peaks = df[df["Peak"] == 3]["Time"].values

        if len(r_peaks) < 2:
            continue

        # RR (ms)
        rr = np.diff(r_peaks) * 1000

        # Clean -> NN
        nn = clean_rr(rr)

        if len(nn) < 5:
            continue

        # Extract features
        mean_rr, hr, sdnn, rmssd, pnn50 = time_domain_features(nn)
        lf, hf, lf_hf = frequency_domain_features(nn)

        results.append({
            "File": filename,
            "Mean_RR(ms)": mean_rr,
            "HR(bpm)": hr,
            "SDNN(ms)": sdnn,
            "RMSSD(ms)": rmssd,
            "pNN50(%)": pnn50,
            "LF Power": lf,
            "HF Power": hf,
            "LF/HF Ratio": lf_hf
        })

# ==============================
# 6. Convert to DataFrame
# ==============================

features_df = pd.DataFrame(results)

print(features_df)

# Optional: save
features_df.to_csv("data\\hrv_features.csv", index=False)

         File  Mean_RR(ms)     HR(bpm)   SDNN(ms)  RMSSD(ms)   pNN50(%)  \
0   hr100.csv   599.715074  100.047510  13.330608   4.932311   0.000000   
1   hr101.csv   593.822674  101.040264  12.918245   4.767402   0.000000   
2   hr102.csv   587.971630  102.045740  12.503373   4.670479   0.000000   
3   hr103.csv   582.200699  103.057245  12.061133   4.550369   0.000000   
4   hr104.csv   576.719811  104.036655  11.805241   4.460409   0.000000   
5   hr105.csv   571.230076  105.036486  11.474376   4.427616   0.000000   
6    hr60.csv   994.949341   60.304578  74.953642  47.969004  32.941176   
7    hr61.csv   978.530649   61.316424  72.032188  45.180965  32.046332   
8    hr62.csv   962.002841   62.369878  67.056307  41.932086  25.475285   
9    hr63.csv   948.256763   63.274002  66.153073  39.621036  24.719101   
10   hr64.csv   933.808379   64.253011  63.436416  37.369985  21.323529   
11   hr65.csv   919.477662   65.254440  60.758649  35.210462  18.478261   
12   hr66.csv   905.62444